> **Data availability:** The raw LabRAM `.txt` spectra are not included in this repository because the final datasets are provided in `data/`. This notebook reads the raw evaluation spectra from `evaluationspectra/`; its processed output is provided as `data/Eval_Preprocessed_aligned_402_3499_recalibrated.xlsx`. This notebook is included to document the processing steps; its file paths refer to the original project layout.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

SRC_DIR = 'evaluationspectra'

TREATMENT_COLOR = {
    'Starch':                  '#1f77b4',
    'Urea':                    '#2ca02c',
    'Dilution':                '#ff7f0e',
    'Vegetable Oil':           '#9467bd',
    'Control (full-fat)':      '#7f7f7f',
}

## 1. File Inventory & Metadata Parsing

In [ ]:
def parse_filename(fname):
    """Extract sample metadata from an evaluation-spectra filename.

    Pattern: '{Concentration/Descriptor} {Treatment} {Brand} {Laser} {Power} {Grating}
               {Objective} {Exposure} {Accumulations} {Range} {Plate}'
    Examples:
      '0pt05% starch Amul 532nm 100% 600g 50xL 3s 10times 200-4000cm-1_96 well plate'
      '3 g_L Urea Amul 532nm 100% 600g 50xL 3s 10times 200-4000cm-1_96 well plate'
      'Akshyakalpa full fat 1 532nm 100% 600g 50xL 3s 10times 200-4000cm-1_96 well plate'
    """
    stem = fname.replace('.txt', '')
    label = stem.split(' 532nm')[0]
    tokens = label.split()
    lower = label.lower()

    if 'akshyakalpa' in lower:
        brand = 'Akshyakalpa'
    elif 'amul' in lower:
        brand = 'Amul'
    else:
        brand = 'unknown'

    if 'starch' in lower:
        treatment = 'Starch'
    elif 'urea' in lower:
        treatment = 'Urea'
    elif 'dilution' in lower:
        treatment = 'Dilution'
    elif 'vegetable oil' in lower:
        treatment = 'Vegetable Oil'
    elif 'full fat' in lower:
        treatment = 'Control (full-fat)'
    else:
        treatment = 'unknown'

    if tokens[0].endswith('%'):
        concentration = tokens[0].replace('pt', '.')
    elif len(tokens) > 1 and tokens[1] == 'g_L':
        concentration = f"{tokens[0].replace('pt', '.')} g/L"
    else:
        concentration = 'n/a'

    stem_tokens = stem.split()
    laser    = next((t for t in stem_tokens if re.match(r'^\d+nm$', t)), 'unknown')
    power    = next((t for t in stem_tokens if t.endswith('%') and t != tokens[0]), 'unknown')
    exposure = next((t for t in stem_tokens if re.match(r'^\d+s$', t)), 'unknown')
    accum    = next((t for t in stem_tokens if t.endswith('times')), 'unknown')
    plate    = 'well plate' if 'well plate' in stem.lower() else 'unknown'

    return {
        'Label':         label,
        'Brand':         brand,
        'Treatment':     treatment,
        'Concentration': concentration,
        'Laser':         laser,
        'Power':         power,
        'Exposure':      exposure,
        'Accumulations': accum,
        'Plate':         plate,
        'Filename':      fname,
    }

all_files = sorted(glob.glob(os.path.join(SRC_DIR, '*.txt')))
meta_rows = [parse_filename(os.path.basename(f)) for f in all_files]
meta_df   = pd.DataFrame(meta_rows)

print(f'Files found: {len(all_files)}')
print()
print(meta_df[['Label', 'Brand', 'Treatment', 'Concentration', 'Laser', 'Power', 'Exposure']].to_string(index=False))

## 2. Load Spectra — Flag Corrupt, Non-Standard-Laser & Grid-Mismatched Files

In [ ]:
def load_spectrum(fpath):
    """Return (wavenumbers, intensities, error) from a LabRAM .txt file."""
    try:
        with open(fpath, 'rb') as fh:
            raw = fh.read()
        text  = raw.decode('latin-1')
        lines = [l for l in text.splitlines()
                 if not l.startswith('#') and l.strip() and '\t' in l]
        if len(lines) == 0:
            return None, None, 'No data lines found'
        pairs = [tuple(map(float, l.split('\t')[:2])) for l in lines]
        wn = np.array([p[0] for p in pairs])
        y  = np.array([p[1] for p in pairs])
        return wn, y, None
    except Exception as e:
        return None, None, str(e)

# ── Pass 1: load everything, determine the common grid length ───────────────
loaded = {}
for fpath in all_files:
    loaded[fpath] = load_spectrum(fpath)

lengths = [len(wn) for wn, y, err in loaded.values() if err is None]
target_len = pd.Series(lengths).mode().iloc[0]
print(f'Common grid length across files: {target_len} points')

# ── Pass 2: classify each file ───────────────────────────────────────────────
spectra_raw = {}
wn_ref      = None
load_log    = []
excluded    = {'corrupt': [], 'non_532nm': [], 'grid_mismatch': []}

for fpath, row in zip(all_files, meta_rows):
    base = os.path.basename(fpath)
    wn, y, err = loaded[fpath]
    label = row['Label']

    if err is not None:
        status = f'CORRUPT — {err[:60]}'
        excluded['corrupt'].append(base)
    elif row['Laser'] != '532nm':
        status = f'EXCLUDED — non-standard laser ({row["Laser"]})'
        excluded['non_532nm'].append(base)
    elif len(wn) != target_len:
        status = f'EXCLUDED — grid mismatch ({len(wn)} pts vs {target_len})'
        excluded['grid_mismatch'].append(base)
    elif label in spectra_raw:
        status = f'EXCLUDED — duplicate label "{label}" (keeping first file)'
        excluded['grid_mismatch'].append(base)
    else:
        if wn_ref is None:
            wn_ref = wn
        spectra_raw[label] = y
        status = f'OK — {len(wn)} pts, wn {wn[0]:.1f}\u2013{wn[-1]:.1f} cm\u207b\u00b9'

    load_log.append({'File': base, 'Label': label, 'Status': status})

load_report = pd.DataFrame(load_log)
print(load_report[['Label', 'Status']].to_string(index=False))
print()
print(f'Successfully loaded : {len(spectra_raw)}')
print(f'Corrupt             : {len(excluded["corrupt"])}')
print(f'Non-532nm laser     : {len(excluded["non_532nm"])}')
print(f'Grid mismatch       : {len(excluded["grid_mismatch"])}')

## 3. Build Raw Combined Dataset

In [ ]:
dataset_raw = pd.DataFrame(spectra_raw, index=wn_ref)
dataset_raw.index.name = 'Raman Shift'

print(f'Raw dataset shape : {dataset_raw.shape[0]} wavenumbers \u00d7 {dataset_raw.shape[1]} spectra')
print(f'Wavenumber range  : {wn_ref.min():.1f} \u2013 {wn_ref.max():.1f} cm\u207b\u00b9')
print(f'Missing values    : {dataset_raw.isna().sum().sum()}')
print()
print('Columns in dataset:')
for col in dataset_raw.columns:
    print(f'  {col}')

## 3b. Restrict to 400–4000 cm⁻¹ Range

The raw acquisition window (200–4000 cm⁻¹) extends below the usable Raman fingerprint region used elsewhere in this project. Trim every spectrum to **400–4000 cm⁻¹** before duplicate detection and preprocessing.

In [ ]:
wn_mask = (dataset_raw.index >= 400) & (dataset_raw.index <= 4000)
n_before = dataset_raw.shape[0]
dataset_raw = dataset_raw.loc[wn_mask]

print(f'Wavenumber points before restriction : {n_before}')
print(f'Wavenumber points after restriction  : {dataset_raw.shape[0]}')
print(f'Restricted range                     : {dataset_raw.index.min():.1f}\u2013{dataset_raw.index.max():.1f} cm\u207b\u00b9')

## 4. Duplicate Detection (Prior to Preprocessing)

Three complementary checks, applied to the raw (pre-preprocessing) intensities:

| Check | Method | Criterion |
|-------|--------|----------|
| **Exact duplicates** | Column-wise equality | All intensity values identical |
| **Near-duplicates (correlation)** | Pearson r between every pair | r > 0.99999 |
| **Near-duplicates (L2 distance)** | L2 distance between every pair | L2 < 1e-6 |

In [ ]:
cols   = list(dataset_raw.columns)
F      = dataset_raw.values
n_spec = len(cols)

# Exact duplicates
exact_dup_pairs = []
for i in range(n_spec):
    for j in range(i + 1, n_spec):
        if np.array_equal(F[:, i], F[:, j]):
            exact_dup_pairs.append((cols[i], cols[j]))
print(f'Exact duplicate pairs found: {len(exact_dup_pairs)}')
for a, b in exact_dup_pairs:
    print(f'  {a}  ==  {b}')

# Near-duplicates by Pearson correlation
CORR_THRESHOLD = 0.99999
corr_matrix = np.corrcoef(F.T)
near_dup_corr = []
for i in range(n_spec):
    for j in range(i + 1, n_spec):
        r = corr_matrix[i, j]
        if r > CORR_THRESHOLD:
            near_dup_corr.append((cols[i], cols[j], round(r, 8)))
print(f'\nNear-duplicate pairs (r > {CORR_THRESHOLD}): {len(near_dup_corr)}')
for a, b, r in near_dup_corr:
    print(f'  {a}  \u2194  {b}   r = {r}')

# Near-duplicates by L2 distance
L2_THRESHOLD = 1e-6
near_dup_l2 = []
for i in range(n_spec):
    for j in range(i + 1, n_spec):
        d = np.linalg.norm(F[:, i] - F[:, j])
        if d < L2_THRESHOLD:
            near_dup_l2.append((cols[i], cols[j], d))
print(f'\nNear-duplicate pairs (L2 < {L2_THRESHOLD}): {len(near_dup_l2)}')
for a, b, d in near_dup_l2:
    print(f'  {a}  \u2194  {b}   L2 = {d:.2e}')

## 5. Remove Duplicates & Build Deduplicated Raw Dataset

In [ ]:
to_drop = set()

for a, b in exact_dup_pairs:
    to_drop.add(b)
    print(f'Exact duplicate \u2014 dropping: {b}  (kept: {a})')

for a, b, r in near_dup_corr:
    if b not in to_drop and a not in to_drop:
        to_drop.add(b)
        print(f'Near-duplicate (r={r:.7f}) \u2014 dropping: {b}  (kept: {a})')

for a, b, d in near_dup_l2:
    if b not in to_drop and a not in to_drop:
        to_drop.add(b)
        print(f'Near-duplicate (L2={d:.2e}) \u2014 dropping: {b}  (kept: {a})')

if not to_drop:
    print('No duplicates detected across all three checks \u2014 all spectra are unique.')

dataset_clean = dataset_raw.drop(columns=list(to_drop))

print(f'\nSpectra before deduplication : {dataset_raw.shape[1]}')
print(f'Duplicates removed           : {len(to_drop)}')
print(f'Spectra in final dataset     : {dataset_clean.shape[1]}')

## 6. Preprocessing Pipeline

Identical order and parameters to `raman_consensus.ipynb`, `raman_qc.ipynb`, `raman_adulterated_dataset.ipynb` and `raman_valid_samples_combined.ipynb`:

1. **Savitzky–Golay smoothing** — window = 5 points, polynomial order = 2
2. **ALS baseline correction** — λ = 1×10⁵, p = 0.01, 10 iterations
3. **Non-negative clip** — set any negative residuals to zero
4. **L² normalisation** — scale each spectrum to unit Euclidean norm

In [ ]:
from scipy.signal import savgol_filter
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

def sg_smooth(y, window=5, polyorder=2):
    return savgol_filter(y, window_length=window, polyorder=polyorder)

def als_baseline(y, lam=1e5, p=0.01, n_iter=10):
    L = len(y)
    D = diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L), format='csc')
    H = lam * D.T.dot(D)
    w = np.ones(L)
    for _ in range(n_iter):
        W = diags(w, 0, format='csc')
        Z = spsolve(W + H, w * y)
        w = np.where(y > Z, p, 1 - p)
    return Z

def l2_normalize(y):
    n = np.linalg.norm(y)
    return y / n if n > 0 else y

def preprocess(y):
    y = sg_smooth(y)
    y = y - als_baseline(y)
    y = np.clip(y, 0, None)
    return l2_normalize(y)

preprocessed = {col: preprocess(dataset_clean[col].values)
                for col in dataset_clean.columns}

dataset_prep = pd.DataFrame(preprocessed, index=dataset_clean.index)
dataset_prep.index.name = 'Raman Shift'

print('Preprocessing complete.')
print(f'Shape  : {dataset_prep.shape}')
print(f'Any NaN: {dataset_prep.isna().any().any()}')
print()
norms = np.linalg.norm(dataset_prep.values, axis=0)
print('L2 norms (should all be 1.0):')
for col, n in zip(dataset_prep.columns, norms):
    print(f'  {col:<30} {n:.6f}')

## 7. Functional Data Representation (B-spline, order 4)

Each preprocessed spectrum is projected onto a cubic B-spline basis (order n = 4, degree = 3) with interior knots placed every 10th wavenumber point — identical settings to the other notebooks. The spline-evaluated spectrum replaces the discrete values.

In [ ]:
from scipy.interpolate import make_lsq_spline

wn_arr = dataset_prep.index.values.astype(float)

interior_knots = wn_arr[10:-10:10]
k = 3  # cubic (order 4)
t = np.concatenate([[wn_arr[0]] * (k + 1), interior_knots, [wn_arr[-1]] * (k + 1)])

discrete_prep = dataset_prep.copy()  # keep the pre-spline values for comparison

spline_fitted = {}
for col in dataset_prep.columns:
    y = dataset_prep[col].values
    spl = make_lsq_spline(wn_arr, y, t=t, k=k)
    spline_fitted[col] = spl(wn_arr)

dataset_prep = pd.DataFrame(spline_fitted, index=dataset_prep.index)
dataset_prep.index.name = 'Raman Shift'

print(f'B-spline (order 4) fit applied to all {dataset_prep.shape[1]} spectra.')
print(f'Interior knots : {len(interior_knots)}  \u2192  total basis functions = {len(t) - k - 1}')

max_dev = np.max(np.abs(dataset_prep.values - discrete_prep.values), axis=0)
print(f'Max abs deviation (spline vs discrete): mean={max_dev.mean():.2e}, max={max_dev.max():.2e}')

## 8. Metadata & Coverage Summary for Final Dataset

In [ ]:
final_cols = list(dataset_clean.columns)
meta_final = meta_df[meta_df['Label'].isin(final_cols)].copy().set_index('Label')

print('Final dataset \u2014 spectrum summary:')
print(f'  Shape   : {dataset_clean.shape[0]} wavenumbers \u00d7 {dataset_clean.shape[1]} spectra')
print(f'  WN range: {dataset_clean.index.min():.1f} \u2013 {dataset_clean.index.max():.1f} cm\u207b\u00b9')
print()
print('Treatment breakdown:')
treatment_counts = meta_final['Treatment'].value_counts()
for treatment, cnt in treatment_counts.items():
    concs = sorted(meta_final[meta_final['Treatment'] == treatment]['Concentration'].tolist())
    print(f'  {treatment:<22}  {cnt} spectra  {concs}')

## 9. Intensity Statistics (Preprocessed)

In [ ]:
stats = pd.DataFrame({
    'Brand':         [meta_final.loc[c, 'Brand'] for c in dataset_prep.columns],
    'Treatment':     [meta_final.loc[c, 'Treatment'] for c in dataset_prep.columns],
    'Concentration': [meta_final.loc[c, 'Concentration'] for c in dataset_prep.columns],
    'Min':           dataset_prep.min().values,
    'Max':           dataset_prep.max().values,
    'Mean':          dataset_prep.mean().values,
    'Std':           dataset_prep.std().values,
    'SNR (mean/std)': (dataset_prep.mean() / dataset_prep.std()).values,
}, index=dataset_prep.columns)

print(stats.round(5).to_string())

## 10. Visualization — All Preprocessed Spectra

In [ ]:
wn = dataset_prep.index.values

fig, ax = plt.subplots(figsize=(14, 5))
for col in dataset_prep.columns:
    treatment = meta_final.loc[col, 'Treatment']
    ax.plot(wn, dataset_prep[col],
            color=TREATMENT_COLOR.get(treatment, 'grey'),
            linewidth=0.9, alpha=0.75)

from matplotlib.patches import Patch
handles = [Patch(color=c, label=t) for t, c in TREATMENT_COLOR.items() if t in treatment_counts.index]
ax.legend(handles=handles, fontsize=9, title='Treatment', loc='upper right')
ax.set_xlabel('Raman Shift (cm\u207b\u00b9)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Evaluation Spectra \u2014 Preprocessed (SG + ALS + L\u00b2 norm + B-spline)')
plt.tight_layout()
plt.show()

## 11. Dataset Construction Report

In [ ]:
print('=' * 70)
print('EVALUATION DATASET CONSTRUCTION REPORT')
print('=' * 70)
print(f'  Source folder        : {SRC_DIR}/')
print(f'  Files discovered     : {len(all_files)}')
print(f'  Corrupt              : {len(excluded["corrupt"])}')
for f in excluded['corrupt']:
    print(f'      \u2717  {f}')
print(f'  Non-532nm laser      : {len(excluded["non_532nm"])}')
for f in excluded['non_532nm']:
    print(f'      \u2717  {f}')
print(f'  Grid mismatch        : {len(excluded["grid_mismatch"])}')
for f in excluded['grid_mismatch']:
    print(f'      \u2717  {f}')
print(f'  Successfully loaded  : {dataset_raw.shape[1]}')
print()
print('  Duplicate checks (pre-preprocessing):')
print(f'    Exact duplicates           : {len(exact_dup_pairs)}')
print(f'    Near-duplicates (r>0.99999): {len(near_dup_corr)}')
print(f'    Near-duplicates (L2<1e-6)  : {len(near_dup_l2)}')
print(f'    Total removed              : {len(to_drop)}')
print()
print(f'  Final dataset : {dataset_clean.shape[1]} spectra \u00d7 {dataset_clean.shape[0]} wavenumbers')
print(f'  WN range      : {dataset_clean.index.min():.1f} \u2013 {dataset_clean.index.max():.1f} cm\u207b\u00b9')
print()
for treatment, cnt in treatment_counts.items():
    concs = sorted(meta_final[meta_final['Treatment'] == treatment]['Concentration'].tolist())
    print(f'  {treatment:<22} : {cnt} spectra at {concs}')
print('=' * 70)

## 12. Save Evaluation Dataset to Excel

In [ ]:
out_path = 'evaluation_spectra_combined.xlsx'

with pd.ExcelWriter(out_path) as writer:
    dataset_clean.to_excel(writer, sheet_name='Spectra Raw')
    dataset_prep.to_excel(writer, sheet_name='Spectra Preprocessed')
    meta_final.reset_index().to_excel(writer, sheet_name='Metadata', index=False)
    stats.to_excel(writer, sheet_name='Statistics')
    load_report.to_excel(writer, sheet_name='Load Log', index=False)

print(f'Saved \u2192 {out_path}')
print(f'  Sheet "Spectra Raw"         : {dataset_clean.shape[1]} spectra \u00d7 {dataset_clean.shape[0]} wn points (raw counts, deduplicated)')
print(f'  Sheet "Spectra Preprocessed": {dataset_prep.shape[1]} spectra \u00d7 {dataset_prep.shape[0]} wn points (SG+ALS+L\u00b2+B-spline)')
print(f'  Sheet "Metadata"            : {len(meta_final)} rows')
print(f'  Sheet "Statistics"          : {len(stats)} rows')
print(f'  Sheet "Load Log"            : {len(load_report)} rows')